# HW10-11: Computer Vision in PyTorch
**Часть A**: CNN, аугментации, transfer learning (STL10)  
**Часть B**: Detection track (Pascal VOC + FasterRCNN)

## 0. Импорты, seed, устройство

In [ ]:
import os
import json
import csv
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset

import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torchvision.datasets import STL10, VOCDetection
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights

# ── Seed ──────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}  |  torchvision: {torchvision.__version__}')

## 1. Часть A — Данные (STL10)

In [ ]:
DATA_ROOT = './data'
os.makedirs(DATA_ROOT, exist_ok=True)

# ── Transforms ────────────────────────────────────────────────────────────────
# STL10: 96×96 px, 10 classes

# базовый (для C1)
transform_base = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4467, 0.4398, 0.4066],
                         std=[0.2242, 0.2215, 0.2239]),
])

# с аугментациями (для C2)
transform_aug = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop(64, padding=8),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4467, 0.4398, 0.4066],
                         std=[0.2242, 0.2215, 0.2239]),
])

# для ResNet pretrained (224×224, ImageNet stats)
transform_resnet_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop(224, padding=28),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

transform_resnet_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# ── Datasets ──────────────────────────────────────────────────────────────────
# STL10: 'train' split = 5000 labeled, 'test' = 8000
# val: отделяем 20% от train (1000 → val, 4000 → train)

full_train_base = STL10(root=DATA_ROOT, split='train', download=True, transform=transform_base)
full_train_aug  = STL10(root=DATA_ROOT, split='train', download=False, transform=transform_aug)
full_train_res  = STL10(root=DATA_ROOT, split='train', download=False, transform=transform_resnet_train)
test_base       = STL10(root=DATA_ROOT, split='test',  download=False, transform=transform_base)
test_res        = STL10(root=DATA_ROOT, split='test',  download=False, transform=transform_resnet_val)

N = len(full_train_base)   # 5000
n_val = int(0.2 * N)       # 1000
n_train = N - n_val        # 4000

g = torch.Generator().manual_seed(SEED)
train_idx, val_idx = torch.randperm(N, generator=g)[:n_train].tolist(), \
                     torch.randperm(N, generator=g)[n_train:N].tolist()
# более аккуратно:
perm = torch.randperm(N, generator=torch.Generator().manual_seed(SEED)).tolist()
train_idx, val_idx = perm[:n_train], perm[n_train:]

def make_loaders(train_ds, val_ds, test_ds, batch_size=64):
    tr = DataLoader(Subset(train_ds, train_idx), batch_size=batch_size,
                    shuffle=True,  num_workers=2, pin_memory=True)
    va = DataLoader(Subset(val_ds,   val_idx),   batch_size=batch_size,
                    shuffle=False, num_workers=2, pin_memory=True)
    te = DataLoader(test_ds, batch_size=batch_size,
                    shuffle=False, num_workers=2, pin_memory=True)
    return tr, va, te

# val всегда без аугментаций
val_base = STL10(root=DATA_ROOT, split='train', download=False, transform=transform_base)
val_res  = STL10(root=DATA_ROOT, split='train', download=False, transform=transform_resnet_val)

loader_c1 = make_loaders(full_train_base, val_base, test_base)
loader_c2 = make_loaders(full_train_aug,  val_base, test_base)   # val без aug
loader_c3 = make_loaders(full_train_res,  val_res,  test_res)
loader_c4 = make_loaders(full_train_res,  val_res,  test_res)

NUM_CLASSES = 10
CLASS_NAMES = full_train_base.classes
print('Classes:', CLASS_NAMES)
print(f'Train: {n_train} | Val: {n_val} | Test: {len(test_base)}')

In [ ]:
# ── Sanity-check ──────────────────────────────────────────────────────────────
x_batch, y_batch = next(iter(loader_c1[0]))
print(f'x.shape = {x_batch.shape}  |  y.shape = {y_batch.shape}  |  dtype = {x_batch.dtype}')
print(f'y sample: {y_batch[:8].tolist()}')

# Визуализация нескольких изображений
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
mean = torch.tensor([0.4467, 0.4398, 0.4066]).view(3,1,1)
std  = torch.tensor([0.2242, 0.2215, 0.2239]).view(3,1,1)
for i, ax in enumerate(axes.flat):
    img = x_batch[i] * std + mean
    ax.imshow(img.permute(1,2,0).clamp(0,1).numpy())
    ax.set_title(CLASS_NAMES[y_batch[i].item()], fontsize=7)
    ax.axis('off')
plt.suptitle('STL10 – sample batch (base transform)', fontsize=10)
plt.tight_layout()
plt.savefig('./artifacts/figures/sanity_check.png', dpi=100, bbox_inches='tight')
plt.show()

## 2. Часть A — Архитектура простой CNN

In [ ]:
class SimpleCNN(nn.Module):
    """Простая CNN для STL10 (64×64 input)."""
    def __init__(self, num_classes=10, dropout=0.4):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 64→32
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),     # 64→32
            nn.Dropout2d(0.1),

            # Block 2: 32→16
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),     # 32→16
            nn.Dropout2d(0.15),

            # Block 3: 16→8
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),     # 16→8
            nn.Dropout2d(0.2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


# Проверка shape
with torch.no_grad():
    dummy = torch.zeros(2, 3, 64, 64)
    out = SimpleCNN()(dummy)
    print(f'SimpleCNN output shape: {out.shape}')  # [2, 10]

total_params = sum(p.numel() for p in SimpleCNN().parameters())
print(f'SimpleCNN total params: {total_params:,}')

## 3. Часть A — Функции обучения и оценки

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, total_correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        total_correct += (logits.argmax(1) == y).sum().item()
        total += x.size(0)
    return total_loss / total, total_correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, total_correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            total_loss += loss.item() * x.size(0)
            total_correct += (logits.argmax(1) == y).sum().item()
            total += x.size(0)
    return total_loss / total, total_correct / total


def run_experiment(model, loaders, name, epochs=30, lr=1e-3, weight_decay=1e-4,
                   scheduler_type='cosine', device=DEVICE):
    """Полный цикл обучения, возвращает history и best_val_acc."""
    train_loader, val_loader, test_loader = loaders
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                            lr=lr, weight_decay=weight_decay)
    if scheduler_type == 'cosine':
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    else:
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_acc = 0.0
    best_state = None

    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        va_loss, va_acc = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(va_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(va_acc)

        if va_acc > best_val_acc:
            best_val_acc = va_acc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        if epoch % 5 == 0 or epoch == 1:
            print(f'[{name}] Epoch {epoch:3d}/{epochs} | '
                  f'tr_loss={tr_loss:.4f} tr_acc={tr_acc:.3f} | '
                  f'va_loss={va_loss:.4f} va_acc={va_acc:.3f}')

    print(f'[{name}] Best val_acc = {best_val_acc:.4f}')
    # load best
    model.load_state_dict(best_state)
    # test (один раз)
    te_loss, te_acc = evaluate(model, test_loader, criterion, device)
    print(f'[{name}] Test acc (best model) = {te_acc:.4f}')

    return history, best_val_acc, te_acc, model, best_state


def plot_history(history, name, save_path=None):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    epochs = range(1, len(history['train_loss']) + 1)
    ax1.plot(epochs, history['train_loss'], label='train')
    ax1.plot(epochs, history['val_loss'],   label='val')
    ax1.set_title(f'{name} – Loss')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend()
    ax2.plot(epochs, history['train_acc'], label='train')
    ax2.plot(epochs, history['val_acc'],   label='val')
    ax2.set_title(f'{name} – Accuracy')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=100, bbox_inches='tight')
    plt.show()

print('Training utilities ready.')

## 4. Визуализация аугментаций

In [ ]:
mean_t = torch.tensor([0.4467, 0.4398, 0.4066]).view(3,1,1)
std_t  = torch.tensor([0.2242, 0.2215, 0.2239]).view(3,1,1)

# Берём одно изображение и применяем aug 8 раз
raw_ds = STL10(root=DATA_ROOT, split='train', download=False, transform=None)
sample_img, sample_lbl = raw_ds[0]

aug_only = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop(64, padding=8),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4467, 0.4398, 0.4066],
                         std=[0.2242, 0.2215, 0.2239]),
])

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
# оригинал
axes[0, 0].imshow(sample_img)
axes[0, 0].set_title('Original', fontsize=9)
axes[0, 0].axis('off')

for i, ax in enumerate(axes.flat[1:]):
    aug_img = aug_only(sample_img) * std_t + mean_t
    ax.imshow(aug_img.permute(1,2,0).clamp(0,1).numpy())
    ax.set_title(f'Aug #{i+1}', fontsize=9)
    ax.axis('off')

plt.suptitle(f'Augmentations preview – class: {CLASS_NAMES[sample_lbl]}', fontsize=11)
plt.tight_layout()
plt.savefig('./artifacts/figures/augmentations_preview.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved: augmentations_preview.png')

## 5. C1 – Simple CNN (без аугментаций)

In [ ]:
torch.manual_seed(SEED)
model_c1 = SimpleCNN(num_classes=NUM_CLASSES)
history_c1, best_val_c1, test_acc_c1, model_c1, state_c1 = run_experiment(
    model_c1, loader_c1, name='C1-SimpleCNN-base',
    epochs=30, lr=1e-3, weight_decay=1e-4, device=DEVICE
)

## 6. C2 – Simple CNN (с аугментациями)

In [ ]:
torch.manual_seed(SEED)
model_c2 = SimpleCNN(num_classes=NUM_CLASSES)
history_c2, best_val_c2, test_acc_c2, model_c2, state_c2 = run_experiment(
    model_c2, loader_c2, name='C2-SimpleCNN-aug',
    epochs=30, lr=1e-3, weight_decay=1e-4, device=DEVICE
)

## 7. C3 – ResNet18 head-only

In [ ]:
torch.manual_seed(SEED)
model_c3 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Замораживаем весь backbone
for param in model_c3.parameters():
    param.requires_grad = False

# Заменяем классификационную голову
in_features = model_c3.fc.in_features   # 512
model_c3.fc = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(in_features, NUM_CLASSES)
)

trainable = sum(p.numel() for p in model_c3.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model_c3.parameters())
print(f'C3: trainable params = {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')

history_c3, best_val_c3, test_acc_c3, model_c3, state_c3 = run_experiment(
    model_c3, loader_c3, name='C3-ResNet18-head-only',
    epochs=20, lr=3e-3, weight_decay=1e-4, device=DEVICE
)

## 8. C4 – ResNet18 partial fine-tune (layer4 + fc)

In [ ]:
torch.manual_seed(SEED)
model_c4 = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# Замораживаем layers 1-3, размораживаем layer4 + fc
for name_p, param in model_c4.named_parameters():
    if 'layer4' in name_p or 'fc' in name_p:
        param.requires_grad = True
    else:
        param.requires_grad = False

# Заменяем fc
in_features = model_c4.fc.in_features
model_c4.fc = nn.Sequential(
    nn.Dropout(0.4),
    nn.Linear(in_features, NUM_CLASSES)
)

trainable = sum(p.numel() for p in model_c4.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model_c4.parameters())
print(f'C4: trainable params = {trainable:,} / {total:,} ({100*trainable/total:.1f}%)')

history_c4, best_val_c4, test_acc_c4, model_c4, state_c4 = run_experiment(
    model_c4, loader_c4, name='C4-ResNet18-finetune',
    epochs=20, lr=5e-4, weight_decay=1e-4, device=DEVICE
)

## 9. Сохранение лучшей модели и артефактов

In [ ]:
results = {
    'C1': (best_val_c1, test_acc_c1, state_c1, model_c1, history_c1, 'SimpleCNN', 'base', 'AdamW', 1e-3, 30),
    'C2': (best_val_c2, test_acc_c2, state_c2, model_c2, history_c2, 'SimpleCNN', 'aug',  'AdamW', 1e-3, 30),
    'C3': (best_val_c3, test_acc_c3, state_c3, model_c3, history_c3, 'ResNet18-head-only', 'aug', 'AdamW', 3e-3, 20),
    'C4': (best_val_c4, test_acc_c4, state_c4, model_c4, history_c4, 'ResNet18-finetune',  'aug', 'AdamW', 5e-4, 20),
}

# Лучший эксперимент по val_accuracy
best_exp = max(results, key=lambda k: results[k][0])
print(f'Лучший эксперимент: {best_exp} | val_acc={results[best_exp][0]:.4f} | test_acc={results[best_exp][1]:.4f}')

# Сохраняем best_classifier.pt
best_state = results[best_exp][2]
torch.save(best_state, './artifacts/best_classifier.pt')
print('Saved: best_classifier.pt')

# Сохраняем best_classifier_config.json
best_config = {
    'experiment_id': best_exp,
    'dataset': 'STL10',
    'num_classes': NUM_CLASSES,
    'architecture': results[best_exp][5],
    'augmentation': results[best_exp][6],
    'optimizer': results[best_exp][7],
    'lr': results[best_exp][8],
    'epochs': results[best_exp][9],
    'seed': SEED,
    'input_size': 224 if 'ResNet' in results[best_exp][5] else 64,
    'normalize_mean': [0.485, 0.456, 0.406] if 'ResNet' in results[best_exp][5] else [0.4467, 0.4398, 0.4066],
    'normalize_std':  [0.229, 0.224, 0.225] if 'ResNet' in results[best_exp][5] else [0.2242, 0.2215, 0.2239],
    'val_split': '20% of train (seed=42)',
    'best_val_accuracy': float(results[best_exp][0]),
    'test_accuracy': float(results[best_exp][1]),
}
with open('./artifacts/best_classifier_config.json', 'w') as f:
    json.dump(best_config, f, indent=2)
print('Saved: best_classifier_config.json')

## 10. Графики кривых и сравнения C1-C4

In [ ]:
# Кривые лучшего прогона
plot_history(results[best_exp][4], name=best_exp,
             save_path='./artifacts/figures/classification_curves_best.png')

# Сравнение C1-C4 – bar plot
exp_names = list(results.keys())
val_accs  = [results[k][0] for k in exp_names]
test_accs = [results[k][1] for k in exp_names]

x = np.arange(len(exp_names))
width = 0.35
fig, ax = plt.subplots(figsize=(8, 5))
bars1 = ax.bar(x - width/2, val_accs,  width, label='val_accuracy',  color='steelblue')
bars2 = ax.bar(x + width/2, test_accs, width, label='test_accuracy', color='tomato', alpha=0.85)

def autolabel(bars):
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f'{h:.3f}', xy=(bar.get_x() + bar.get_width()/2, h),
                    xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=9)
autolabel(bars1)
autolabel(bars2)

ax.set_title('Comparison C1–C4: val & test accuracy (STL10)', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(exp_names)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Accuracy')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('./artifacts/figures/classification_compare.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved: classification_compare.png')

## 11. Часть B – Detection (Pascal VOC + FasterRCNN)

In [ ]:
# VOC class mapping (20 классов)
VOC_CLASSES = [
    '__background__', 'aeroplane', 'bicycle', 'bird', 'boat', 'bottle',
    'bus', 'car', 'cat', 'chair', 'cow', 'diningtable', 'dog', 'horse',
    'motorbike', 'person', 'pottedplant', 'sheep', 'sofa', 'train', 'tvmonitor'
]
# COCO classes для FasterRCNN (91 класс)
COCO_CLASSES = [
    '__background__', 'person', 'bicycle', 'car', 'motorcycle', 'airplane',
    'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'N/A',
    'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse',
    'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'N/A', 'backpack',
    'umbrella', 'N/A', 'N/A', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis',
    'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove',
    'skateboard', 'surfboard', 'tennis racket', 'bottle', 'N/A', 'wine glass',
    'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich',
    'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake',
    'chair', 'couch', 'potted plant', 'bed', 'N/A', 'dining table', 'N/A',
    'N/A', 'toilet', 'N/A', 'tv', 'laptop', 'mouse', 'remote', 'keyboard',
    'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator',
    'N/A', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier',
    'toothbrush'
]

# Маппинг VOC → COCO label id
VOC_TO_COCO = {
    'aeroplane': 5, 'bicycle': 2, 'bird': 16, 'boat': 9, 'bottle': 44,
    'bus': 6, 'car': 3, 'cat': 17, 'chair': 62, 'cow': 21,
    'diningtable': 67, 'dog': 18, 'horse': 19, 'motorbike': 4, 'person': 1,
    'pottedplant': 64, 'sheep': 20, 'sofa': 63, 'train': 7, 'tvmonitor': 72
}

print('VOC detection setup ready.')

In [ ]:
# Загрузка Pascal VOC 2007 (detection)
voc_ds = VOCDetection(
    root=DATA_ROOT,
    year='2007',
    image_set='val',       # validation set
    download=True,
    transform=None,
)
print(f'VOC val samples: {len(voc_ds)}')

# Sanity-check
img0, ann0 = voc_ds[0]
print(f'Image size: {img0.size}')  # PIL Image
print(f'Annotation keys: {ann0["annotation"].keys()}')
objs = ann0['annotation']['object']
if isinstance(objs, dict): objs = [objs]
print(f'Objects in sample[0]: {[(o["name"], o["bndbox"]) for o in objs]}')

In [ ]:
# Загрузка pretrained FasterRCNN
det_model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
det_model = det_model.to(DEVICE)
det_model.eval()

to_tensor = transforms.ToTensor()


def get_predictions(img_pil, score_threshold=0.5):
    """Инференс FasterRCNN на одном PIL-изображении."""
    img_t = to_tensor(img_pil).to(DEVICE).unsqueeze(0)
    with torch.no_grad():
        preds = det_model(img_t)[0]
    mask = preds['scores'] >= score_threshold
    return {
        'boxes':  preds['boxes'][mask].cpu(),
        'labels': preds['labels'][mask].cpu(),
        'scores': preds['scores'][mask].cpu(),
    }


def parse_gt(ann):
    """Парсит аннотацию VOC в списки boxes и labels (VOC-names)."""
    objs = ann['annotation']['object']
    if isinstance(objs, dict): objs = [objs]
    boxes, labels = [], []
    for obj in objs:
        bb = obj['bndbox']
        x1, y1, x2, y2 = float(bb['xmin']), float(bb['ymin']), float(bb['xmax']), float(bb['ymax'])
        boxes.append([x1, y1, x2, y2])
        labels.append(obj['name'])
    return torch.tensor(boxes, dtype=torch.float32), labels


print('FasterRCNN loaded.')

In [ ]:
def compute_iou(box1, box2):
    """IoU между двумя boxes [x1,y1,x2,y2]."""
    x1 = max(box1[0], box2[0]); y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2]); y2 = min(box1[3], box2[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    a1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    a2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    return inter / (a1 + a2 - inter + 1e-8)


def evaluate_detection(dataset, score_threshold, n_samples=200, iou_threshold=0.5):
    """Базовая оценка detection: precision, recall, mean_iou."""
    tp, fp, fn_count = 0, 0, 0
    matched_ious = []

    indices = list(range(min(n_samples, len(dataset))))
    for idx in indices:
        img, ann = dataset[idx]
        preds = get_predictions(img, score_threshold=score_threshold)
        gt_boxes, gt_labels = parse_gt(ann)

        pred_boxes  = preds['boxes']
        pred_labels = preds['labels']

        matched_gt = set()
        for pb, pl in zip(pred_boxes, pred_labels):
            pred_name = COCO_CLASSES[pl.item()] if pl.item() < len(COCO_CLASSES) else ''
            best_iou, best_j = 0.0, -1
            for j, (gb, gl) in enumerate(zip(gt_boxes, gt_labels)):
                if j in matched_gt: continue
                # проверяем совпадение класса (VOC-name ≈ COCO-name, нечёткое)
                if pred_name.lower() not in gl.lower() and gl.lower() not in pred_name.lower():
                    # приблизительное сопоставление через VOC_TO_COCO
                    coco_id = VOC_TO_COCO.get(gl, -1)
                    if coco_id != pl.item():
                        continue
                iou = compute_iou(pb.tolist(), gb.tolist())
                if iou > best_iou:
                    best_iou, best_j = iou, j
            if best_iou >= iou_threshold and best_j >= 0:
                tp += 1
                matched_gt.add(best_j)
                matched_ious.append(best_iou)
            else:
                fp += 1
        fn_count += len(gt_boxes) - len(matched_gt)

    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn_count + 1e-8)
    mean_iou  = float(np.mean(matched_ious)) if matched_ious else 0.0

    return {'tp': tp, 'fp': fp, 'fn': fn_count,
            'precision': precision, 'recall': recall,
            'mean_iou': mean_iou, 'n_matched': len(matched_ious)}


print('Detection evaluation utilities ready.')

In [ ]:
# V1: score_threshold = 0.3
print('Running V1 (threshold=0.3)...')
metrics_v1 = evaluate_detection(voc_ds, score_threshold=0.3, n_samples=200)
print(f"V1 | precision={metrics_v1['precision']:.4f} | recall={metrics_v1['recall']:.4f} | mean_iou={metrics_v1['mean_iou']:.4f}")
print(f"    tp={metrics_v1['tp']}  fp={metrics_v1['fp']}  fn={metrics_v1['fn']}  matched={metrics_v1['n_matched']}")

In [ ]:
# V2: score_threshold = 0.7
print('Running V2 (threshold=0.7)...')
metrics_v2 = evaluate_detection(voc_ds, score_threshold=0.7, n_samples=200)
print(f"V2 | precision={metrics_v2['precision']:.4f} | recall={metrics_v2['recall']:.4f} | mean_iou={metrics_v2['mean_iou']:.4f}")
print(f"    tp={metrics_v2['tp']}  fp={metrics_v2['fp']}  fn={metrics_v2['fn']}  matched={metrics_v2['n_matched']}")

## 12. Визуализация detection предсказаний

In [ ]:
def visualize_detections(dataset, score_threshold, n=6, title_prefix='V1'):
    """Рисует bounding boxes pred (красный) и GT (зелёный) для n изображений."""
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    for ax, idx in zip(axes.flat, range(n)):
        img, ann = dataset[idx]
        preds = get_predictions(img, score_threshold=score_threshold)
        gt_boxes, gt_labels = parse_gt(ann)

        ax.imshow(img)
        # GT – зелёный
        for (x1,y1,x2,y2), lbl in zip(gt_boxes.tolist(), gt_labels):
            rect = patches.Rectangle((x1,y1), x2-x1, y2-y1,
                                      linewidth=2, edgecolor='lime', facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y1-4, lbl, color='lime', fontsize=7, weight='bold')
        # Pred – красный
        for (x1,y1,x2,y2), pl, sc in zip(
                preds['boxes'].tolist(), preds['labels'].tolist(), preds['scores'].tolist()):
            cls_name = COCO_CLASSES[pl] if pl < len(COCO_CLASSES) else str(pl)
            rect = patches.Rectangle((x1,y1), x2-x1, y2-y1,
                                      linewidth=2, edgecolor='red', facecolor='none')
            ax.add_patch(rect)
            ax.text(x1, y2+12, f'{cls_name} {sc:.2f}', color='red', fontsize=6)
        ax.set_title(f'Sample {idx} | thr={score_threshold}', fontsize=8)
        ax.axis('off')

    plt.suptitle(f'{title_prefix} Detection: FasterRCNN on Pascal VOC 2007\n'
                 f'Green=GT, Red=Pred (thr={score_threshold})', fontsize=11)
    plt.tight_layout()
    fname = f'./artifacts/figures/detection_examples_{title_prefix.lower()}.png'
    plt.savefig('./artifacts/figures/detection_examples.png', dpi=100, bbox_inches='tight')
    plt.show()
    print('Saved: detection_examples.png')

visualize_detections(voc_ds, score_threshold=0.5, n=6, title_prefix='V1+V2 combined')

In [ ]:
# Detection metrics comparison: V1 vs V2
metrics_keys = ['precision', 'recall', 'mean_iou']
v1_vals = [metrics_v1[k] for k in metrics_keys]
v2_vals = [metrics_v2[k] for k in metrics_keys]

x = np.arange(len(metrics_keys))
width = 0.35
fig, ax = plt.subplots(figsize=(8, 5))
bars_v1 = ax.bar(x - width/2, v1_vals, width, label='V1 (thr=0.3)', color='steelblue')
bars_v2 = ax.bar(x + width/2, v2_vals, width, label='V2 (thr=0.7)', color='coral')

def autolabel2(bars):
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f'{h:.3f}', xy=(bar.get_x() + bar.get_width()/2, h),
                    xytext=(0,3), textcoords='offset points', ha='center', va='bottom', fontsize=9)
autolabel2(bars_v1); autolabel2(bars_v2)

ax.set_title('Detection metrics: V1 vs V2 (Pascal VOC 2007, n=200)', fontsize=12)
ax.set_xticks(x); ax.set_xticklabels(metrics_keys)
ax.set_ylim(0, 1.1); ax.set_ylabel('Score')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('./artifacts/figures/detection_metrics.png', dpi=100, bbox_inches='tight')
plt.show()
print('Saved: detection_metrics.png')

## 13. Сохранение runs.csv

In [ ]:
fieldnames = [
    'experiment_id', 'task', 'dataset', 'seed', 'model_summary',
    'optimizer', 'lr', 'epochs_trained', 'best_val_accuracy',
    'test_accuracy', 'precision', 'recall', 'mean_iou', 'notes'
]

rows = [
    {
        'experiment_id': 'C1', 'task': 'classification', 'dataset': 'STL10',
        'seed': SEED, 'model_summary': 'SimpleCNN (3-block, ~1.3M params)',
        'optimizer': 'AdamW', 'lr': 1e-3, 'epochs_trained': 30,
        'best_val_accuracy': round(best_val_c1, 4),
        'test_accuracy': round(test_acc_c1, 4),
        'precision': '', 'recall': '', 'mean_iou': '',
        'notes': 'base transform, no augmentations'
    },
    {
        'experiment_id': 'C2', 'task': 'classification', 'dataset': 'STL10',
        'seed': SEED, 'model_summary': 'SimpleCNN (3-block, ~1.3M params)',
        'optimizer': 'AdamW', 'lr': 1e-3, 'epochs_trained': 30,
        'best_val_accuracy': round(best_val_c2, 4),
        'test_accuracy': round(test_acc_c2, 4),
        'precision': '', 'recall': '', 'mean_iou': '',
        'notes': 'flip, crop, colorjitter, rotation'
    },
    {
        'experiment_id': 'C3', 'task': 'classification', 'dataset': 'STL10',
        'seed': SEED, 'model_summary': 'ResNet18 pretrained, head-only (fc frozen backbone)',
        'optimizer': 'AdamW', 'lr': 3e-3, 'epochs_trained': 20,
        'best_val_accuracy': round(best_val_c3, 4),
        'test_accuracy': round(test_acc_c3, 4),
        'precision': '', 'recall': '', 'mean_iou': '',
        'notes': 'ImageNet pretrained, only fc trained'
    },
    {
        'experiment_id': 'C4', 'task': 'classification', 'dataset': 'STL10',
        'seed': SEED, 'model_summary': 'ResNet18 pretrained, layer4+fc fine-tuned',
        'optimizer': 'AdamW', 'lr': 5e-4, 'epochs_trained': 20,
        'best_val_accuracy': round(best_val_c4, 4),
        'test_accuracy': round(test_acc_c4, 4),
        'precision': '', 'recall': '', 'mean_iou': '',
        'notes': 'layer4+fc unfrozen, lower lr'
    },
    {
        'experiment_id': 'V1', 'task': 'detection', 'dataset': 'Pascal VOC 2007',
        'seed': SEED, 'model_summary': 'FasterRCNN_ResNet50_FPN (COCO pretrained)',
        'optimizer': 'N/A', 'lr': 'N/A', 'epochs_trained': 0,
        'best_val_accuracy': '',
        'test_accuracy': '',
        'precision': round(metrics_v1['precision'], 4),
        'recall':    round(metrics_v1['recall'],    4),
        'mean_iou':  round(metrics_v1['mean_iou'],  4),
        'notes': 'score_threshold=0.3, IoU_thr=0.5, n=200 samples'
    },
    {
        'experiment_id': 'V2', 'task': 'detection', 'dataset': 'Pascal VOC 2007',
        'seed': SEED, 'model_summary': 'FasterRCNN_ResNet50_FPN (COCO pretrained)',
        'optimizer': 'N/A', 'lr': 'N/A', 'epochs_trained': 0,
        'best_val_accuracy': '',
        'test_accuracy': '',
        'precision': round(metrics_v2['precision'], 4),
        'recall':    round(metrics_v2['recall'],    4),
        'mean_iou':  round(metrics_v2['mean_iou'],  4),
        'notes': 'score_threshold=0.7, IoU_thr=0.5, n=200 samples'
    },
]

with open('./artifacts/runs.csv', 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print('Saved: runs.csv')

# Итоговая сводка
print('\n=== FINAL SUMMARY ===')
for r in rows:
    print(r)